# 1. Setup Spark

In [1]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 3.5 MB/s eta 0:00:0000:0100:02
  Preparing metadata (setup.py) ... done
  Obtaining dependency information for py4j<0.10.9.10,>=0.10.9.7 from https://files.pythonhosted.org/packages/bd/db/ea0203e495be491c85af87b66e37acfd3bf756fd985f87e46fc5e3bf022c/py4j-0.10.9.9-py2.py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 2.1 MB/s eta 0:00:00ta 0:00:01
  Created wheel for pyspark: filename=pyspark-4.1.1-py2.py3-none-any.whl size=456008650 sha256=d990da5eb507b46631a24e41f9013ed758e09ccbdf20fd2ac1d63d7b957d4c08
  Stored in directory: /Users/enfants/Library/Caches/pip/wheels/16/33/a9/f8bff354a182417214933df74dace2a34b02c3e5643e8fac74
Successfully built pyspark


In [ ]:
import pyspark
import pyspark.sql
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Predicting U.S. Flight Delay") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/07 09:37:47 WARN Utils: Your hostname, MacBook-Air-de-Nicolas-3.local, resolves to a loopback address: 127.0.0.1; using 192.0.0.2 instead (on interface en8)
26/03/07 09:37:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 09:37:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# 2. Loading Data

In [ ]:
flights = spark.read.csv(
    "Data/Flights/*.csv",
    header=True,
    inferSchema=True
)

26/03/07 09:37:49 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: Data/Flights/*.csv.
java.io.FileNotFoundException: File Data/Flights/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveDa

Row(FL_DATE=datetime.date(2013, 7, 1), OP_CARRIER_AIRLINE_ID=20363, OP_CARRIER_FL_NUM=3407, ORIGIN_AIRPORT_ID=11433, DEST_AIRPORT_ID=13342, CRS_DEP_TIME=1040, ARR_DELAY_NEW=0.0, CANCELLED=0.0, DIVERTED=0.0, CRS_ELAPSED_TIME=79.0, WEATHER_DELAY=None, NAS_DELAY=None, _c12=None)

In [6]:
flights.printSchema()
flights.show(5)

root
 |-- FL_DATE: date (nullable = true)
 |-- OP_CARRIER_AIRLINE_ID: integer (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- ARR_DELAY_NEW: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- CRS_ELAPSED_TIME: double (nullable = true)
 |-- WEATHER_DELAY: double (nullable = true)
 |-- NAS_DELAY: double (nullable = true)
 |-- _c12: string (nullable = true)

+----------+---------------------+-----------------+-----------------+---------------+------------+-------------+---------+--------+----------------+-------------+---------+----+
|   FL_DATE|OP_CARRIER_AIRLINE_ID|OP_CARRIER_FL_NUM|ORIGIN_AIRPORT_ID|DEST_AIRPORT_ID|CRS_DEP_TIME|ARR_DELAY_NEW|CANCELLED|DIVERTED|CRS_ELAPSED_TIME|WEATHER_DELAY|NAS_DELAY|_c12|
+----------+---------------------+-----------------+----

26/03/07 09:37:57 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: FL_DATE, OP_CARRIER_AIRLINE_ID, OP_CARRIER_FL_NUM, ORIGIN_AIRPORT_ID, DEST_AIRPORT_ID, CRS_DEP_TIME, ARR_DELAY_NEW, CANCELLED, DIVERTED, CRS_ELAPSED_TIME, WEATHER_DELAY, NAS_DELAY, 
 Schema: FL_DATE, OP_CARRIER_AIRLINE_ID, OP_CARRIER_FL_NUM, ORIGIN_AIRPORT_ID, DEST_AIRPORT_ID, CRS_DEP_TIME, ARR_DELAY_NEW, CANCELLED, DIVERTED, CRS_ELAPSED_TIME, WEATHER_DELAY, NAS_DELAY, _c12
Expected: _c12 but found: 
CSV file: file:///Users/enfants/Code/Predicting%20U.S.%20Flight%20Delays/Data/Flights/201307.csv


In [7]:
weather = spark.read.csv(
    "Data/Weather/*.txt",
    header=True,
    inferSchema=True
)

26/03/07 09:37:57 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: Data/Weather/*.txt.
java.io.FileNotFoundException: File Data/Weather/*.txt does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.ResolveDa

In [8]:
weather.printSchema()
weather.show(5)

root
 |-- WBAN: integer (nullable = true)
 |-- Date: integer (nullable = true)
 |-- Time: integer (nullable = true)
 |-- StationType: integer (nullable = true)
 |-- SkyCondition: string (nullable = true)
 |-- SkyConditionFlag: string (nullable = true)
 |-- Visibility: string (nullable = true)
 |-- VisibilityFlag: string (nullable = true)
 |-- WeatherType: string (nullable = true)
 |-- WeatherTypeFlag: string (nullable = true)
 |-- DryBulbFarenheit: string (nullable = true)
 |-- DryBulbFarenheitFlag: string (nullable = true)
 |-- DryBulbCelsius: string (nullable = true)
 |-- DryBulbCelsiusFlag: string (nullable = true)
 |-- WetBulbFarenheit: string (nullable = true)
 |-- WetBulbFarenheitFlag: string (nullable = true)
 |-- WetBulbCelsius: string (nullable = true)
 |-- WetBulbCelsiusFlag: string (nullable = true)
 |-- DewPointFarenheit: string (nullable = true)
 |-- DewPointFarenheitFlag: string (nullable = true)
 |-- DewPointCelsius: string (nullable = true)
 |-- DewPointCelsiusFlag: str

26/03/07 09:38:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [9]:
print("Flights:", flights.count())
print("Weather:", weather.count())

Flights: 18286055


Weather: 32631312


In [ ]:
wban_airport_timezone = spark.read.csv(
    "Data/wban_airport_timezone.csv",
    header=True,
    inferSchema=True
)

In [ ]:
wban_airport_timezone.printSchema()
wban_airport_timezone.show(5)

root
 |-- AirportID: integer (nullable = true)
 |-- WBAN: integer (nullable = true)
 |-- TimeZone: integer (nullable = true)

+---------+-----+--------+
|AirportID| WBAN|TimeZone|
+---------+-----+--------+
|    10685|54831|      -6|
|    14871|24232|      -8|
|    10620|24033|      -7|
|    14747|24233|      -8|
|    11252|12834|      -5|
+---------+-----+--------+
only showing top 5 rows


# 3. Filter and clean

In [50]:
from typing import Iterable, Optional
from pyspark.sql import DataFrame, functions as F


def first_existing(df: DataFrame, candidates: Iterable[str]) -> Optional[str]:
    cols = set(df.columns)
    for c in candidates:
        if c in cols:
            return c
    return None


def normalized_date_string(date_col: str):
    raw = F.trim(F.col(date_col).cast("string"))
    return (
        F.when(raw.rlike(r"^\d{4}-\d{2}-\d{2}$"), raw)
         .when(raw.rlike(r"^\d{8}$"),
               F.concat_ws("-", F.substring(raw, 1, 4), F.substring(raw, 5, 2), F.substring(raw, 7, 2)))
         .when(raw.rlike(r"^\d{2}/\d{2}/\d{4}$"),
               F.concat_ws("-", F.substring(raw, 7, 4), F.substring(raw, 1, 2), F.substring(raw, 4, 2)))
    )


def hhmm_to_timestamp(date_col: str, hhmm_col: str):
    date_str = normalized_date_string(date_col)
    raw = F.regexp_replace(F.trim(F.col(hhmm_col).cast("string")), r"[^0-9]", "")
    hhmm = F.lpad(raw, 4, "0")

    is_2400 = hhmm == F.lit("2400")
    hh = F.when(is_2400, F.lit("00")).otherwise(F.substring(hhmm, 1, 2))
    mm = F.when(is_2400, F.lit("00")).otherwise(F.substring(hhmm, 3, 2))

    base_ts = F.to_timestamp(
        F.concat_ws(" ", date_str, F.concat_ws(":", hh, mm, F.lit("00"))),
        "yyyy-MM-dd HH:mm:ss",
    )

    return F.when(is_2400, base_ts + F.expr("INTERVAL 1 DAY")).otherwise(base_ts)


def add_timestamp_from_candidates(
    df: DataFrame,
    out_col: str,
    date_candidates: Iterable[str],
    time_candidates: Iterable[str],
) -> DataFrame:
    date_col = first_existing(df, date_candidates)
    time_col = first_existing(df, time_candidates)
    if not (date_col and time_col):
        return df
    return df.withColumn(out_col, hhmm_to_timestamp(date_col, time_col))


def build_ft(flights_df: DataFrame) -> DataFrame:
    ft = flights_df

    cancelled_col = first_existing(ft, ["CANCELLED", "Cancelled"])
    diverted_col = first_existing(ft, ["DIVERTED", "Diverted"])
    if cancelled_col:
        ft = ft.filter(F.coalesce(F.col(cancelled_col).cast("double"), F.lit(0.0)) == 0.0)
    if diverted_col:
        ft = ft.filter(F.coalesce(F.col(diverted_col).cast("double"), F.lit(0.0)) == 0.0)

    origin_col = first_existing(ft, ["ORIGIN_AIRPORT_ID", "ORIGIN_AIRPORT_SEQ_ID", "ORIGIN"])
    destination_col = first_existing(ft, ["DEST_AIRPORT_ID", "DEST_AIRPORT_SEQ_ID", "DEST"])

    if origin_col:
        ft = ft.withColumn("origin_airport_id", F.col(origin_col).cast("int"))
    if destination_col:
        ft = ft.withColumn("destination_airport_id", F.col(destination_col).cast("int"))

    ft = add_timestamp_from_candidates(ft, "scheduled_departure_time", ["FL_DATE", "FLIGHT_DATE"], ["CRS_DEP_TIME"])
    ft = add_timestamp_from_candidates(ft, "actual_departure_time", ["FL_DATE", "FLIGHT_DATE"], ["DEP_TIME"])
    ft = add_timestamp_from_candidates(ft, "scheduled_arrival_time", ["FL_DATE", "FLIGHT_DATE"], ["CRS_ARR_TIME"])
    ft = add_timestamp_from_candidates(ft, "actual_arrival_time", ["FL_DATE", "FLIGHT_DATE"], ["ARR_TIME"])

    arrival_delay_col = first_existing(ft, ["ARR_DELAY_NEW", "ARR_DELAY"])
    if arrival_delay_col:
        ft = ft.withColumn("arrival_delay_minutes", F.col(arrival_delay_col).cast("double"))

    arr_del15_col = first_existing(ft, ["ARR_DEL15"])
    if arr_del15_col:
        ft = ft.withColumn("delay_label", F.col(arr_del15_col).cast("int"))
    elif "arrival_delay_minutes" in ft.columns:
        ft = ft.withColumn(
            "delay_label",
            F.when(F.col("arrival_delay_minutes") >= 15, F.lit(1)).otherwise(F.lit(0)),
        )

    return ft


def build_ot(weather_df: DataFrame, mapping_df: DataFrame) -> DataFrame:
    weather_wban_col = first_existing(weather_df, ["WBAN"])
    mapping_wban_col = first_existing(mapping_df, ["WBAN"])
    mapping_airport_col = first_existing(mapping_df, ["AirportID", "AIRPORT_ID"])

    if not (weather_wban_col and mapping_wban_col and mapping_airport_col):
        return weather_df

    airport_mapping = (
        mapping_df
        .select(
            F.col(mapping_wban_col).cast("string").alias("map_wban"),
            F.col(mapping_airport_col).cast("int").alias("airport_id"),
            *([F.col("TimeZone").alias("time_zone")] if "TimeZone" in mapping_df.columns else []),
        )
        .dropna(subset=["map_wban", "airport_id"])
        .dropDuplicates(["map_wban", "airport_id"])
    )

    ot = (
        weather_df
        .withColumn("weather_wban", F.col(weather_wban_col).cast("string"))
        .join(airport_mapping, F.col("weather_wban") == F.col("map_wban"), "inner")
        .drop("weather_wban", "map_wban")
    )

    ot = add_timestamp_from_candidates(
        ot,
        "observation_time",
        ["DATE", "Date", "YEARMODA", "OBS_DATE"],
        ["TIME", "Time", "OBS_TIME"],
    )

    rename_map = {
        "DryBulbCelsius": "dry_bulb_celsius",
        "RelativeHumidity": "relative_humidity",
        "WindDirection": "wind_direction_degrees",
        "WindSpeed": "wind_speed",
        "StationPressure": "station_pressure",
        "SeaLevelPressure": "sea_level_pressure",
        "SkyCondition": "sky_condition",
        "Visibility": "visibility",
        "WeatherType": "weather_type",
        "HourlyPrecip": "hourly_precipitation",
    }

    for src, dst in rename_map.items():
        if src in ot.columns and dst not in ot.columns:
            ot = ot.withColumnRenamed(src, dst)

    if "observation_time" in ot.columns:
        ot = ot.dropna(subset=["observation_time"]).dropDuplicates(["airport_id", "observation_time"])

    return ot


FT = build_ft(flights)
OT = build_ot(weather, wban_airport_timezone)

In [51]:
FT.show(5)
OT.show(5)
print("FT:", FT.count())
print("OT:", OT.count())

26/03/07 12:05:41 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: FL_DATE, OP_CARRIER_AIRLINE_ID, OP_CARRIER_FL_NUM, ORIGIN_AIRPORT_ID, DEST_AIRPORT_ID, CRS_DEP_TIME, ARR_DELAY_NEW, CANCELLED, DIVERTED, CRS_ELAPSED_TIME, WEATHER_DELAY, NAS_DELAY, 
 Schema: FL_DATE, OP_CARRIER_AIRLINE_ID, OP_CARRIER_FL_NUM, ORIGIN_AIRPORT_ID, DEST_AIRPORT_ID, CRS_DEP_TIME, ARR_DELAY_NEW, CANCELLED, DIVERTED, CRS_ELAPSED_TIME, WEATHER_DELAY, NAS_DELAY, _c12
Expected: _c12 but found: 
CSV file: file:///Users/enfants/Code/Predicting%20U.S.%20Flight%20Delays/Data/Flights/201307.csv


+----------+---------------------+-----------------+-----------------+---------------+------------+-------------+---------+--------+----------------+-------------+---------+----+----------------------+------------------------+---------------------+-----------+
|   FL_DATE|OP_CARRIER_AIRLINE_ID|OP_CARRIER_FL_NUM|origin_airport_id|DEST_AIRPORT_ID|CRS_DEP_TIME|ARR_DELAY_NEW|CANCELLED|DIVERTED|CRS_ELAPSED_TIME|WEATHER_DELAY|NAS_DELAY|_c12|destination_airport_id|scheduled_departure_time|arrival_delay_minutes|delay_label|
+----------+---------------------+-----------------+-----------------+---------------+------------+-------------+---------+--------+----------------+-------------+---------+----+----------------------+------------------------+---------------------+-----------+
|2013-07-01|                20363|             3407|            11433|          13342|        1040|          0.0|      0.0|     0.0|            79.0|         NULL|     NULL|NULL|                 13342|     2013-07-01 

+-----+--------+----+-----------+-------------+----------------+----------+--------------+------------+---------------+----------------+--------------------+----------------+------------------+----------------+--------------------+--------------+------------------+-----------------+---------------------+---------------+-------------------+-----------------+--------------------+----------+-------------+----------------------+-----------------+---------------------+-------------------------+----------------+-------------------+----------------+--------------------+--------------+------------------+------------------+--------------------+----------+--------------+--------------------+----------------+---------+-------------+-----------+-------------------+----------+---------+-------------------+
| WBAN|    Date|Time|StationType|sky_condition|SkyConditionFlag|visibility|VisibilityFlag|weather_type|WeatherTypeFlag|DryBulbFarenheit|DryBulbFarenheitFlag|dry_bulb_celsius|DryBulbCelsiusFlag|We

FT: 17943069


OT: 2407742


26/03/07 21:01:49 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 2049982 ms exceeds timeout 120000 ms
26/03/07 21:01:49 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/07 21:01:51 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$

# Join

In [ ]:
# Optional: build a first Joint Table aligned on origin airport + scheduled departure hour.
ft_for_join = FT
ot_for_join = OT

if "scheduled_departure_time" in FT.columns:
    ft_for_join = ft_for_join.withColumn(
        "scheduled_departure_hour",
        F.date_trunc("hour", F.col("scheduled_departure_time")),
    )

if "observation_time" in OT.columns:
    ot_for_join = ot_for_join.withColumn(
        "observation_hour",
        F.date_trunc("hour", F.col("observation_time")),
    )

if {"origin_airport_id", "scheduled_departure_hour"}.issubset(set(ft_for_join.columns)) and {"airport_id", "observation_hour"}.issubset(set(ot_for_join.columns)):
    joint_table = (
        ft_for_join.alias("f")
        .join(
            ot_for_join.alias("o"),
            (F.col("f.origin_airport_id") == F.col("o.airport_id"))
            & (F.col("f.scheduled_departure_hour") == F.col("o.observation_hour")),
            "left",
        )
    )
    joint_table.show(5)
    print("Joint table:", joint_table.count())
else:
    print("Joint table not built: required columns are missing.")

